In [13]:
import concurrent.futures
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
import json
import requests
from datetime import datetime, timedelta
import concurrent.futures
import time

# **Asynchronous Weather Data Fetching Script Explanation**  

## **Overview**  
This script **fetches weather forecast data** (temperature and rainfall) asynchronously from the Open-Meteo API for multiple ecoregions. It then processes the data and saves the **hottest, coldest, and wettest places** to CSV files.  

The script uses **`asyncio` and `aiohttp`** for efficient parallel execution, significantly improving speed compared to traditional synchronous requests.  

---

### **1.Enabling Nested Async (For Jupyter)**  
```python
nest_asyncio.apply()
```
- This ensures **Jupyter Notebook** can run `asyncio` properly by preventing event loop conflicts.  

---

### **2.Flattening the JSON Data**  
```python
coordinates = [
    {"region": region_name, "latitude": region_data["centroid"][0], "longitude": region_data["centroid"][1]}
    for region_type, regions in data.items()
    for region_name, region_data in regions.items()
]
```
- Converts the JSON structure into a **list of dictionaries** for easier processing (using .items() function that return dict_item datatype).  
- Each dictionary contains:
  - `"region"` → Name of the ecoregion.  
  - `"latitude"` and `"longitude"` → Geographical coordinates.  

---

### **3.Fetching Weather Forecast Data Asynchronously**  
- **Asynchronous function** that fetches **daily max & min temperature and rainfall** for a given location.  
- Uses **Open-Meteo API**, requesting **daily temperature and precipitation data**.  

### **4.Handling API Rate Limits and Errors**  
- Uses **`session.get()`** for an asynchronous API request.  
- **Rate limit handling**:  
  - If **HTTP 429 (Too Many Requests)** is returned, it **waits (`await asyncio.sleep()`)** before retrying.  
  - Retries up to **3 times** before giving up.   

### **5.Processing the API Response**  
- Extracts **temperature and rainfall data** from the API response.  
- Returns a dictionary containing:  
  - `"region"` → Name of the ecoregion.  
  - `"max_temperature"` → Maximum daily temperature.  
  - `"min_temperature"` → Minimum daily temperature.  
  - `"max_rainfall"` → Maximum daily precipitation.  

### **6.Handling Request Errors**  
```python
        except Exception as e:
            print(f"Error for {coord['region']}: {e}")
    return None
```
- Logs any **network or API failures**.  
- Returns `None` if no valid data is retrieved after **3 retries**.  

---

## Processing All Coordinates Asynchronously**  

- **Creates an `aiohttp.ClientSession()`** for managing multiple API requests efficiently.  
- Uses **list comprehension** to create a list of tasks (`fetch_forecast_data(session, coord)`).  
- `asyncio.gather(*tasks, return_exceptions=True)` → **Runs all requests in parallel**.  
- Filters out `None` values (failed requests).  

---

## ** Main Function: Sorting and Saving Results**  
- Calls `process_all_coordinates()` to **fetch weather data** asynchronously.  
- Converts the results into a **pandas DataFrame**.  

## **Key Features of the Script**  
✅ Uses **asynchronous requests** for faster execution.  
✅ Implements **rate limit handling** to prevent API failures.  
✅ Sorts and **extracts the hottest, coldest, and wettest places**.  



In [1]:
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
import json

nest_asyncio.apply()

# Load JSON file
file_path = "../Ecoregions_Coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Flatten data
# .items function returns a sort of list (dict_items data type that is non viewable that gives a iterable but not printable list of key-value pairs as tuples, so at top level I get biome and associated dictionaries) 
coordinates = [
    {"region": region_name, "latitude": region_data["centroid"][0], "longitude": region_data["centroid"][1]}
    for region_type, regions in data.items()
    for region_name, region_data in regions.items()
]

# Fetch data for temperature and rainfall
async def fetch_forecast_data(session, coord):
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": coord["latitude"],
        "longitude": coord["longitude"],
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "timezone": "UTC",
    }

    retries = 3
    for attempt in range(retries):
        try:
            async with session.get(BASE_URL, params=params, timeout=15) as response:
                if response.status == 429:
                    retry_after = int(response.headers.get("Retry-After", 5))
                    await asyncio.sleep(retry_after)
                    continue
                response.raise_for_status()
                data = await response.json()
                if "daily" in data:
                    return {
                        "region": coord["region"],
                        "max_temperature": data["daily"]["temperature_2m_max"][0],
                        "min_temperature": data["daily"]["temperature_2m_min"][0],
                        "max_rainfall": data["daily"]["precipitation_sum"][0],
                    }
        except Exception as e:
            print(f"Error for {coord['region']}: {e}")
    return None

# Process all coordinates
async def process_all_coordinates():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_forecast_data(session, coord) for coord in coordinates]
        results = await asyncio.gather(*tasks, return_exceptions=True)
    return [res for res in results if res]

# Main function
async def main():
    results = await process_all_coordinates()
    df = pd.DataFrame(results)

    hottest = df.sort_values(by="max_temperature", ascending=False).head(100)
    coldest = df.sort_values(by="min_temperature").head(100)
    wettest = df.sort_values(by="max_rainfall", ascending=False).head(100)

    hottest.to_csv("hottest_places.csv", index=False)
    coldest.to_csv("coldest_places.csv", index=False)
    wettest.to_csv("wettest_places.csv", index=False)

# Run the async function
await main()


# Linking hottest, coldest and wettest places into a [Database, places.db](places.db)
--> Primary and Foreign Key is rankings

In [2]:
filepath = 'coldest_places.csv'
df_coldest_places=pd.read_csv(filepath)


# Drop the 'max_temperature' column, as places are only ranked based on lowest temperature reached! (for ranking purposes)
df_coldest_places = df_coldest_places.drop(columns=['max_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_coldest_places = pd.merge(df_coldest_places, coordinates_df, on="region", how="left") # Left merge to retain df_coldest_places columns while adding coordinates for locations in top 100 where there is a match
df_coldest_places = df_coldest_places.rename(columns={'min_temperature': 'temperature'}) 
df_coldest_places.to_csv('coldest_places.csv', index=False) # replace original CSV with updated one with coordinates!
print(df_coldest_places)

                                               region  temperature  \
0                     Great Lakes Basin desert steppe        -36.6   
1                     Khangai Mountains alpine meadow        -29.9   
2          Alberta-British Columbia foothills forests        -29.2   
3                           Sayan Intermontane steppe        -29.1   
4   North Tibetan Plateau-Kunlun Mountains alpine ...        -28.9   
..                                                ...          ...   
95                        Gissaro-Alai open woodlands         -4.7   
96                         Wyoming Basin shrub steppe         -4.6   
97                East Afghan montane conifer forests         -4.6   
98                      Atlantic coastal pine barrens         -4.6   
99                       Southern Great Lakes forests         -4.4   

    max_rainfall   latitude   longitude  
0            0.0  48.741138   93.684901  
1            0.0  47.495290   99.304744  
2            0.4  55.051769 -117.

In [3]:
filepath = 'hottest_places.csv'
df_hottest_places=pd.read_csv(filepath)


# Drop the 'min_temperature' column, only interested in max_temperature as that is basis of comparison
df_hottest_places = df_hottest_places.drop(columns=['min_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_hottest_places = pd.merge(df_hottest_places, coordinates_df, on="region", how="left")
df_hottest_places = df_hottest_places.rename(columns={'max_temperature': 'temperature'})
print(df_hottest_places)
df_hottest_places.to_csv('hottest_places.csv', index=False)

                                        region  temperature  max_rainfall  \
0                   Carnarvon xeric shrublands         50.3           0.0   
1                                Gibson desert         45.7           0.0   
2                    Great Sandy-Tanami desert         45.3           0.0   
3          Western Australian Mulga shrublands         44.8           0.0   
4                        Great Victoria desert         44.6           0.0   
..                                         ...          ...           ...   
95                        Sinaloan dry forests         32.2           0.0   
96                Myanmar coastal rain forests         32.2           0.0   
97  South Deccan Plateau dry deciduous forests         32.2           0.5   
98                   Congolian coastal forests         32.1           3.5   
99               Galápagos Islands xeric scrub         32.1           0.0   

     latitude   longitude  
0  -24.312842  114.569487  
1  -24.346783  125.

In [6]:
filepath = 'wettest_places.csv'
df_wettest_places=pd.read_csv(filepath)


df_wettest_places = df_wettest_places.drop(columns=["max_temperature", "min_temperature"]) #only interested in keeping max_rainfall
df_wettest_places = pd.merge(df_wettest_places, coordinates_df, on="region", how="left")
df_wettest_places.to_csv("wettest_places.csv", index=False)

# Initial Exploration of Map Creation

## Choosing Between `geopandas` and `folium`

During the initial exploration, I considered using either `geopandas` or `folium` for mapping. After deliberation and reading about the features of both, I realized that `folium` was the better choice for creating **interactive** maps.  

We wanted to generate **dynamic** maps, which `geopandas` does not support. Instead, `geopandas` is more suited for static visualizations and geospatial data analysis.  

One key motivation for using `folium` was my use of **forecasted data**—I wanted to visualize how Pokémon move across the world as certain places get **warmer** and others get **cooler**. `folium`'s support for animations, interactive layers, and popups made it the ideal choice.

---

## Basic Functionality and Map Features

In the section below, I explore the basic functions of `folium`, which Adrian and Hailey further develop later in **[XXX]()**.  

The key steps include:  

1. **Converting Coordinates to Map Points**  
   - Extracting the latitude and longitude of the **hottest** and **coldest** places we identified.  
   - Plotting them on a map using `folium.Marker()`.  

2. **Adding Tooltips and Icons**  
   - Creating **tooltips** (hoverable text) to display location details.  
   - Using different **icons** to represent various points of interest.  

3. **Visualizing Temperature Ranges**  
   - Representing temperature variations using **color intensity**.  
   - Implementing a **heatmap** to illustrate how temperature changes dynamically over time.  

---

In [ ]:
import folium
from folium.plugins import HeatMap
from branca.colormap import linear

In [4]:
# Initialize a base map (centered at a midpoint between the hottest and coldest locations)
m = folium.Map(location=[df_hottest_places['latitude'].mean(), df_hottest_places['longitude'].mean()], zoom_start=2)

# Add markers for hottest places
for _, row in df_hottest_places.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='red', icon='cloud')
    ).add_to(m)

# Add markers for coldest places
for _, row in df_coldest_places.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='blue', icon='cloud')
    ).add_to(m)

# Save or display the map
m.save("temperature_map.html")
m


In [5]:
import folium
from branca.colormap import linear
import pandas as pd





# Initialise the base map centered between hottest and coldest places
m = folium.Map(location=[(df_hottest_places['latitude'].mean() + df_coldest_places['latitude'].mean()) / 2,
                         (df_hottest_places['longitude'].mean() + df_coldest_places['longitude'].mean()) / 2], zoom_start=2)

# Create color scales for hot and cold places
hot_colormap = linear.Reds_09.scale(df_hottest_places['temperature'].min(), df_hottest_places['temperature'].max())
cold_colormap = linear.Blues_09.scale(df_coldest_places['temperature'].min(), df_coldest_places['temperature'].max())

# Add markers for hottest places with gradient colors
for idx, row in df_hottest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='red', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=hot_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add markers for coldest places with gradient colors
for idx, row in df_coldest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='blue', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=cold_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add legends to the map
hot_colormap.caption = 'Temperature Scale (Hottest Places)'
cold_colormap.caption = 'Temperature Scale (Coldest Places)'
hot_colormap.add_to(m)
cold_colormap.add_to(m)

# Save or display the map
m.save("temperature_ranked_map.html")
m


In [30]:
# Initialise the base map centered at an average location
m = folium.Map(location=[(df_hottest_places['latitude'].mean() + df_coldest_places['latitude'].mean()) / 2,
                         (df_hottest_places['longitude'].mean() + df_coldest_places['longitude'].mean()) / 2], zoom_start=2)

# Create a color scale for temperatures
colormap = linear.RdYlBu_11.scale(df_coldest_places['temperature'].min(), df_hottest_places['temperature'].max())

# Function to map temperature to color intensity
def get_marker_color(temp, temp_min, temp_max, color_scale):
    # Map temperature to a hex color
    hex_color = color_scale(temp)
    return hex_color

# Add markers for hottest places
for _, row in df_hottest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap),
        fill_opacity=0.8
    ).add_to(m)

# Add markers for coldest places
for _, row in df_coldest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap),
        fill_opacity=0.8
    ).add_to(m)

# Add color scale legend to map
colormap.caption = 'Temperature Scale (°C)'
colormap.add_to(m)

# Add HeatMap for temperature intensity
heat_data = [[row['latitude'], row['longitude'], row['temperature']] 
             for _, row in pd.concat([df_hottest_places, df_coldest_places]).iterrows()]
HeatMap(heat_data).add_to(m)

# Save or display the map
m.save("temperature_map_with_scale.html")
m


# Creation of Database!
* Note that pre-creation of database df.sort_values() function was used during collection of data from [openmeteo](https://open-meteo.com/), so data in [coldest_places.csv](coldest_places.csv), [hottest_places.csv](hottest_places.csv) and [wettest_places.csv](wettest_places.csv) is already ordered from highest rank to lowest rank, so all that is left is to rank them accordingly using an additional rankings column.

In [1]:
import sqlite3


# Load CSV files
coldest_places = pd.read_csv("coldest_places.csv")
hottest_places = pd.read_csv("hottest_places.csv")
wettest_places = pd.read_csv("wettest_places.csv") 

# Sort coldest places and assign unique ranking
coldest_places = coldest_places.sort_values(by="temperature", ascending=True)
coldest_places["ranking"] = range(1, len(coldest_places) + 1) # range(1,101) is used for rankings to be generated from 1-100, quirok of range function:)

# Sort hottest places and assign unique ranking
hottest_places = hottest_places.sort_values(by="temperature", ascending=False)
hottest_places["ranking"] = range(1, len(hottest_places) + 1)

# Sort wettest places and assign unique ranking
wettest_places = wettest_places.sort_values(by="max_rainfall", ascending=False)
wettest_places["ranking"] = range(1, len(wettest_places) + 1)

# Connect to SQLite database
conn = sqlite3.connect("places.db")
cursor = conn.cursor()

# Drop existing tables if they exist
cursor.executescript("""
DROP TABLE IF EXISTS hottest_places;
DROP TABLE IF EXISTS coldest_places;
DROP TABLE IF EXISTS wettest_places;

CREATE TABLE hottest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    temperature FLOAT,
    latitude FLOAT,
    longitude FLOAT
);

CREATE TABLE coldest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    temperature FLOAT,
    latitude FLOAT,
    longitude FLOAT
);

CREATE TABLE wettest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    max_rainfall FLOAT,
    latitude FLOAT,
    longitude FLOAT
);
""")

# Insert data into tables
coldest_places[["ranking", "region", "temperature", "latitude", "longitude"]].to_sql("coldest_places", conn, if_exists="append", index=False)
hottest_places[["ranking", "region", "temperature", "latitude", "longitude"]].to_sql("hottest_places", conn, if_exists="append", index=False)
wettest_places[["ranking", "region", "max_rainfall", "latitude", "longitude"]].to_sql("wettest_places", conn, if_exists="append", index=False)

# Commit and close connection
conn.commit()
conn.close()



NameError: name 'pd' is not defined